# DriveSense-VLM — 05: Quick Start (Full Pipeline)

**GPU**: A100 80GB | **Time**: ~8–10 h | **Cost**: ~110 CU

Runs the **entire DriveSense-VLM pipeline** end-to-end in one notebook:
data → training → optimization → benchmarks → evaluation.

| Stage | GPU | CU |
|-------|-----|----|
| Data pipeline | A100 | ~5 |
| SFT training | A100 (~6 h) | ~72 |
| Optimization | A100 (~1.5 h) | ~18 |
| Benchmarks | A100 (~1 h) | ~12 |
| Evaluation | A100 (~1 h) | ~12 |
| **Total** | | **~119** |

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **RECOVERY**: All checkpoints and outputs are on Google Drive.
> If Colab disconnects, rerun Cells 2–3, then rerun the stage that was interrupted.
> Every stage is idempotent (skips if output already exists).

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install ALL dependencies for the full pipeline
# IMPORTANT: Do NOT install numpy or pandas — use Colab's pre-installed versions.
# nuscenes-devkit installed with --no-deps to skip its strict numpy version constraint.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install nuscenes-devkit --no-deps -q 2>&1 | tail -1
!pip install pyquaternion matplotlib tqdm Pillow pyyaml requests openpyxl -q 2>&1 | tail -1
!pip install scikit-learn scipy pyspark -q 2>&1 | tail -1
!pip install transformers peft accelerate datasets bitsandbytes wandb -q 2>&1 | tail -1
!pip install autoawq -q 2>&1 | tail -1
!pip install flash-attn --no-build-isolation -q 2>&1 | tail -3
# TensorRT: best-effort
!pip install tensorrt -q 2>&1 | tail -1 || echo "TensorRT not available — torch.compile fallback"

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import drivesense, numpy, pandas
print(f"✓ numpy   {numpy.__version__}")
print(f"✓ pandas  {pandas.__version__}")
print(f"✓ drivesense importable")

In [ ]:
# GPU verify + checkpoint check + W&B login
import os, glob, torch, wandb

assert torch.cuda.is_available(), "No GPU!"

# Check existing training checkpoints
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
os.makedirs(TRAIN_OUT, exist_ok=True)
ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
if ckpts:
    print(f"Found {len(ckpts)} existing checkpoint(s) — training will resume.")
    print(f"  Latest: {ckpts[-1]}")
else:
    print("No checkpoints — will start fresh training.")

# W&B login (optional — comment out to skip)
wandb.login()
print("✓ W&B configured")

## Stage 1: Data Pipeline

Filters nuScenes frames by rarity score, builds unified dataset, and creates SFT JSONL.
Uses `--mock-llm` for annotation (no API key needed). For real annotation, set `ANTHROPIC_API_KEY`.

In [ ]:
import os, json

# ── nuScenes rarity filtering ──────────────────────────────────────
FILTER_OUTPUT = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
NUSCENES_DIR  = f"{DATA_ROOT}/nuscenes"
os.makedirs(FILTER_OUTPUT, exist_ok=True)

if os.path.exists(f"{FILTER_OUTPUT}/metadata.jsonl"):
    print("✓ Filtering already done — skipping.")
else:
    !python scripts/run_nuscenes_filter.py \
        --nuscenes-root {NUSCENES_DIR} \
        --version v1.0-mini \
        --output-dir {FILTER_OUTPUT} \
        --min-score 1 \
        2>&1 | tail -5

    # CRITICAL: JSON array → JSONL conversion
    json_path  = f"{FILTER_OUTPUT}/metadata.json"
    jsonl_path = f"{FILTER_OUTPUT}/metadata.jsonl"
    if os.path.exists(json_path) and not os.path.exists(jsonl_path):
        with open(json_path) as f:
            frames = json.load(f)
        with open(jsonl_path, 'w') as f:
            for frame in frames:
                f.write(json.dumps(frame) + '\n')
        print(f"  ✓ Converted {len(frames)} frames → metadata.jsonl")

# ── PySpark ETL ────────────────────────────────────────────────────
SPARK_OUTPUT = f"{OUTPUTS_ROOT}/data/spark_processed"
if not (os.path.exists(SPARK_OUTPUT) and os.listdir(SPARK_OUTPUT)):
    os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-11-openjdk-amd64')
    !python scripts/run_spark_pipeline.py \
        --version v1.0-mini \
        --nuscenes-root {DATA_ROOT}/nuscenes \
        --output-dir {SPARK_OUTPUT} \
        2>&1 | tail -3

# ── Unified dataset ────────────────────────────────────────────────
UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
DADA_OUTPUT    = f"{OUTPUTS_ROOT}/data/dada_extracted"
os.makedirs(UNIFIED_OUTPUT, exist_ok=True)
if not os.path.exists(f"{UNIFIED_OUTPUT}/manifest.json"):
    dada_flag = f"--dada-dir {DADA_OUTPUT}" if os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl") else "--nuscenes-only"
    !python scripts/run_build_unified_dataset.py \
        --nuscenes-dir {FILTER_OUTPUT} \
        --output-dir {UNIFIED_OUTPUT} \
        {dada_flag} \
        2>&1 | tail -5

# ── Annotation (mock) ──────────────────────────────────────────────
ANNOTATED_DIR = f"{OUTPUTS_ROOT}/data/annotated"
CACHE_DIR     = f"{OUTPUTS_ROOT}/data/annotation_cache"
SFT_DIR       = f"{OUTPUTS_ROOT}/data/sft_ready"
os.makedirs(ANNOTATED_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# For real annotation: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
USE_MOCK  = 'ANTHROPIC_API_KEY' not in os.environ
mock_flag = "--mock-llm" if USE_MOCK else ""

if not os.path.exists(f"{ANNOTATED_DIR}/annotated_manifest.json"):
    !python scripts/run_annotation_pipeline.py \
        --unified-dir {UNIFIED_OUTPUT} \
        --output-dir {ANNOTATED_DIR} \
        --cache-dir {CACHE_DIR} \
        {mock_flag} \
        2>&1 | tail -5

# ── SFT formatting ─────────────────────────────────────────────────
os.makedirs(SFT_DIR, exist_ok=True)
if not os.path.exists(f"{SFT_DIR}/sft_train.jsonl"):
    !python scripts/run_annotation_pipeline.py \
        --format-only \
        --annotated-dir {ANNOTATED_DIR} \
        --output-dir {SFT_DIR} \
        2>&1 | tail -5

# Summary
print("\n=== Stage 1 Complete ===")
for split in ("train", "val", "test"):
    p = f"{SFT_DIR}/sft_{split}.jsonl"
    if os.path.exists(p):
        with open(p) as f: n = sum(1 for _ in f)
        print(f"  {split}: {n} examples")

## Stage 2: SFT Training

Checkpoints saved to Drive every 250 steps. Auto-resumes after disconnect.

In [ ]:
import os

SFT_DIR   = f"{OUTPUTS_ROOT}/data/sft_ready"
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

# Copy to fast local SSD for DataLoader I/O
!cp {SFT_DIR}/sft_train.jsonl /content/sft_train.jsonl 2>/dev/null || true
!cp {SFT_DIR}/sft_val.jsonl   /content/sft_val.jsonl   2>/dev/null || true
print("✓ SFT data on fast local storage")

# RECOVERY: If session disconnects, rerun Cells 2-3 then rerun this cell.
!python scripts/run_training.py \
    --config configs/training.yaml \
    --override training.output_dir={TRAIN_OUT} \
    --override training.save_steps=250 \
    --override training.save_total_limit=3 \
    --resume \
    2>&1
print("\n=== Stage 2 Complete ===")

## Stage 3: Model Optimization

LoRA merge → AWQ quantization → TensorRT ViT. Each stage is idempotent.

In [ ]:
import os, glob

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

if best and os.path.exists(best):
    !python scripts/run_optimize_model.py --all \
        --adapter-path {best} \
        --override inference.merge.output_dir={OUTPUTS_ROOT}/merged_model \
        --override inference.quantization.output_dir={OUTPUTS_ROOT}/quantized_model \
        --override inference.tensorrt.output_dir={OUTPUTS_ROOT}/tensorrt \
        2>&1 | tail -10
else:
    print("No checkpoint found — running mock optimization for demo.")
    !python scripts/run_optimize_model.py --all --mock \
        --override inference.merge.output_dir={OUTPUTS_ROOT}/merged_model \
        --override inference.quantization.output_dir={OUTPUTS_ROOT}/quantized_model \
        --override inference.tensorrt.output_dir={OUTPUTS_ROOT}/tensorrt \
        2>&1 | tail -5
print("\n=== Stage 3 Complete ===")

## Stage 4: Inference Benchmarks

In [ ]:
import os

BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
QUANT_DIR = f"{OUTPUTS_ROOT}/quantized_model"
os.makedirs(BENCH_DIR, exist_ok=True)
MOCK_FLAG = "" if os.path.exists(QUANT_DIR) else "--mock"

!python scripts/run_benchmark.py --vllm \
    --output {BENCH_DIR}/vllm_bench.json {MOCK_FLAG} 2>&1 | tail -5
!python scripts/run_benchmark.py --local \
    --output {BENCH_DIR}/local_bench.json {MOCK_FLAG} 2>&1 | tail -5
!python scripts/run_benchmark.py --vit-only \
    --output {BENCH_DIR}/vit_bench.json {MOCK_FLAG} 2>&1 | tail -5
print("\n=== Stage 4 Complete ===")

## Stage 5: Evaluation

In [ ]:
import os, glob

PRED_DIR  = f"{OUTPUTS_ROOT}/predictions"
PRED_FILE = f"{PRED_DIR}/test_predictions.jsonl"
EVAL_DIR  = f"{OUTPUTS_ROOT}/eval"
BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

# Locate checkpoint for prediction generation
best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

adapter_flag = f"--adapter-path {best}" if best and os.path.exists(best) else "--mock"
mock_flag    = "--mock" if "--mock" in adapter_flag else ""

# Generate predictions
if not os.path.exists(PRED_FILE):
    !python scripts/run_generate_predictions.py \
        --split test {adapter_flag} \
        --output {PRED_FILE} {mock_flag} 2>&1 | tail -5

# All 4 evaluation levels
!python scripts/run_full_evaluation.py \
    --level 1 2 3 4 \
    --mock-judge \
    --benchmarks-dir {BENCH_DIR} \
    --predictions {PRED_FILE} \
    --output-dir {EVAL_DIR} \
    {mock_flag} \
    2>&1 | tail -15

# Final report
!python scripts/run_full_evaluation.py \
    --generate-report \
    --output-dir {EVAL_DIR} \
    2>&1 | tail -5

# Display report
for candidate in [f"{EVAL_DIR}/final_report.txt", f"{EVAL_DIR}/evaluation_report.txt"]:
    if os.path.exists(candidate):
        print(open(candidate).read())
        break

print("\n=== Pipeline Complete ===")
print(f"All outputs at: {OUTPUTS_ROOT}")
print("\nNext steps:")
print("  1. Fill real metric values into MODEL_CARD.md")
print("  2. Push model to HuggingFace Hub")
print("  3. Deploy demo/app.py to HF Spaces (T4 GPU)")